In [ ]:
import editdistance, tokenize, io
from typing import List

# Tokenize Python code into tokens
def tokenize_py(code: str):
    toks=[]
    # loop through code tokens
    for tok in tokenize.generate_tokens(io.StringIO(code).readline):
        # only keep names, numbers, strings, and operators
        if tok.type in (tokenize.NAME, tokenize.NUMBER, tokenize.STRING, tokenize.OP):
            # append token string to list
            toks.append(tok.string)
    return toks

# calculate levenshtein similarity between two strings
def lev_similarity(a: str, b: str):
    # split code by whitespace separator
    La, Lb = a.split(), b.split()

    # compute normalized Levenshtein similarity
    ed = editdistance.eval(La, Lb)

    # normalize by length of longer snippet
    max_len = max(len(La), len(Lb), 1)
    sim = 1 - ed / max_len

    # keep values to [0,1]
    return max(0.0, min(1.0, sim))

# calculate lexical similarity scores between two code snippets
def lexical_scores(a: str, b: str):
    # compute normalized Levenshtein similarity
    lev_sim = lev_similarity(a, b)

    # get token lists
    ta, tb = tokenize_py(a), tokenize_py(b)

    # compute Jaccard token overlap
    overlap = len(set(ta) & set(tb)) / max(1, len(set(ta) | set(tb)))

    # return dictionary of scores
    return {"lev": lev_sim, "tok_jacc": overlap}


In [24]:
import ast

# get ast node type names from code
def ast_shape(code: str):
    try:
        # parse code into AST
        tree = ast.parse(code)
        # return list of AST node type names
        return [type(n).__name__ for n in ast.walk(tree)]
    except Exception:
        return ["PARSE_ERROR"]

# compute Jaccard similarity between AST node type sets of two code snippets
def ast_jaccard(a: str, b: str):
    # get AST node type sets
    A, B = set(ast_shape(a)), set(ast_shape(b))
    # return Jaccard similarity of AST node type sets
    return len(A & B) / max(1, len(A | B))

In [25]:
from sentence_transformers import SentenceTransformer
import numpy as np
_model = SentenceTransformer("microsoft/codebert-base")

# embed code snippet into vector
def embed(code: str):
    return _model.encode([code])[0]

# calculate cosine similarity between two vectors
def cos(a, b): 
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

No sentence-transformers model found with name microsoft/codebert-base. Creating a new one with mean pooling.


In [26]:
# import subprocess, tempfile, pathlib, json, textwrap, os

# TEST_TMPL = """
# import pytest
# from candidate import {fname}

# def test_sample_1():
#     assert True  # TODO: plug real tests; for now smoke
# """

# def run_tests_on_candidate(candidate_path: str):
#     # create temp directory
#     with tempfile.TemporaryDirectory() as td:
#         td = pathlib.Path(td)
#         # write candidate as candidate.py
#         cand_code = open(candidate_path).read()
#         (td/"candidate.py").write_text(cand_code)
        
#         # very minimal test file
#         (td/"test_candidate.py").write_text(TEST_TMPL.format(fname="*"))
        
#         # run pytest
#         r = subprocess.run(["pytest", "-q", "--maxfail=1", "--disable-warnings", "."] , cwd=td, capture_output=True, text=True, timeout=20)
#         passed = "1 passed" in r.stdout
        
#         # return results
#         return {"passed": passed, "stdout": r.stdout[-500:], "returncode": r.returncode}


In [27]:
def compare_against_candidates(code_A_path: str, cand_paths: List[str]):
    # read code A and embed
    a = open(code_A_path).read()
    a_emb = embed(a)

    out=[]

    # loop through candidate paths
    for cp in cand_paths:
        # read candidate code
        b = open(cp).read()
        
        # compute scores
        scores = {}
        scores.update(lexical_scores(a, b))
        scores["ast"] = ast_jaccard(a,b)
        scores["sem"] = cos(a_emb, embed(b))

        # run tests on candidate
        # dyn = run_tests_on_candidate(cp)
        # scores["dyn_pass"] = 1.0 if dyn["passed"] else 0.0

        # append candidate path and scores to output list
        out.append({"candidate": cp, "scores": scores})
    
    # return output list
    return out

In [ ]:
# --- Test cell for Notebook 03 ---

import os, json, glob, pathlib

# 1) Load a spec (first one) and locate its candidates
spec_files = sorted(glob.glob("specs/*.json"))
assert spec_files, "No spec JSON found in ./specs. Run Notebook 01 first."
with open(spec_files[0], "r", encoding="utf-8") as f:
    spec_data = json.load(f)

# get snippet id
sid = spec_data.get("snippet_id")
assert sid, "Spec JSON has no 'snippet_id'."

# locate candidates for this spec
cand_paths = sorted(glob.glob(f"candidates/{sid}__*.py"))
assert cand_paths, f"No candidates found for spec {sid}. Run Notebook 02 first."

print(f"[✓] Loaded spec: {os.path.basename(spec_files[0])}  (sid={sid})")
print(f"[✓] Found {len(cand_paths)} candidate(s) for this spec.")

# 2) Prepare original snippet for comparison
orig_dir = pathlib.Path("original")
orig_dir.mkdir(exist_ok=True)
orig_path = orig_dir / f"{sid}.py"

if orig_path.exists():
    print(f"[✓] Using original snippet at: {orig_path}")
else:
    print(f"[!] No original/{sid}.py found. Creating a fallback factorial snippet for smoke test.")
    orig_path.write_text(
        """def factorial(n):
    if n < 0:
        raise ValueError("Negative not allowed")
    if n == 0:
        return 1
    out = 1
    for i in range(1, n+1):
        out *= i
    return out
    """,
        encoding="utf-8",
    )

# 3) Compare against candidates
per_candidate = compare_against_candidates(str(orig_path), cand_paths)
print(f"[✓] Results computed for {len(per_candidate)} candidates.")
print(f"[✓] Sample candidate scores:\n{json.dumps(per_candidate[0], indent=2)}")

# 4) Fusion using weighted sum of scores
# _default_weights = {"lev":0.15, "tok_jacc":0.15, "ast":0.20, "sem":0.40, "dyn_pass":0.10}
_default_weights = {"lev": 0.15, "tok_jacc": 0.15, "ast": 0.25, "sem": 0.45}

def _fallback_fuse(d, weights=_default_weights):
    return sum(weights.get(k, 0.0) * d.get(k, 0.0) for k in weights)

# _use_fuse = "fuse" in globals() and callable(globals()["fuse"])
for pc in per_candidate:
    pc["scores"]["fused"] = _fallback_fuse(pc["scores"])

# 5) Pretty summary (sorted by fused desc)
try:
    import pandas as pd
    df = pd.DataFrame(
        [
            {
                "candidate": os.path.basename(pc["candidate"]),
                "lev": round(pc["scores"]["lev"], 4),
                "tok_jacc": round(pc["scores"]["tok_jacc"], 4),
                "ast": round(pc["scores"]["ast"], 4),
                "sem": round(pc["scores"]["sem"], 4),
                # "dyn_pass": int(pc["scores"]["dyn_pass"]),
                "fused": round(pc["scores"]["fused"], 4),
            }
            for pc in per_candidate
        ]
    ).sort_values("fused", ascending=False)
    print("\n[Summary — top candidates by fused score]")
    display(df)
except Exception:
    # fallback plain print
    per_candidate_sorted = sorted(per_candidate, key=lambda x: x["scores"]["fused"], reverse=True)
    print("\n[Summary — top candidates by fused score]")
    for pc in per_candidate_sorted:
        s = pc["scores"]
        print(f"- {os.path.basename(pc['candidate'])} | fused={s['fused']:.4f} | sem={s['sem']:.4f} | ast={s['ast']:.4f} | lev={s['lev']:.4f} | tok_jacc={s['tok_jacc']:.4f} | dyn={int(s['dyn_pass'])}")

# 6) Save detailed report
os.makedirs("reports", exist_ok=True)
report_path = f"reports/{sid}_similarity.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(per_candidate, f, indent=2)
print(f"\n[✓] Saved detailed per-candidate scores to: {report_path}")


[✓] Loaded spec: e2ed22abfca3.json  (sid=e2ed22abfca3)
[✓] Found 4 candidate(s) for this spec.
[✓] Using original snippet at: original/e2ed22abfca3.py
[✓] Results computed for 4 candidates.
[✓] Sample candidate scores:
{
  "candidate": "candidates/e2ed22abfca3__openai:gpt-4o-mini__t0.0__k0.py",
  "scores": {
    "lev": 0.4117647058823529,
    "tok_jacc": 0.6470588235294118,
    "ast": 0.7777777777777778,
    "sem": 0.991959810256958
  }
}

[Summary — top candidates by fused score]


,candidate,lev,tok_jacc,ast,sem,fused
1,e2ed22abfca3__openai:gpt-4o-mini__t0.0__k1.py,0.4118,0.6471,0.7778,0.9925,0.7999
0,e2ed22abfca3__openai:gpt-4o-mini__t0.0__k0.py,0.4118,0.6471,0.7778,0.9920,0.7996
3,e2ed22abfca3__openai:gpt-4o-mini__t0.7__k1.py,0.2909,0.6562,0.8333,0.9937,0.7976
2,e2ed22abfca3__openai:gpt-4o-mini__t0.7__k0.py,0.2759,0.6562,0.8333,0.9925,0.7948



[✓] Saved detailed per-candidate scores to: reports/e2ed22abfca3_similarity.json
